In [1]:
import pandas as pd
import duckdb

df_device = pd.DataFrame({
    "device_id": ["R05", "R16", "R34", "R36", "R40"],
    "device_name": [
        "前向散射仪",
        "云高仪",
        "跑道视程仪",
        "自动气象站",
        "温湿度传感器"
    ]
})

df_alarm = pd.DataFrame({
    "alarm_id": [
        101, 102,
        201,
        301, 302,
        401,
        501
    ],
    "device_id": [
        "R05", "R05",
        "R16",
        "R34", "R34",
        "R36",
        "R40"
    ],
    "alarm_level": [
        "ERROR", "WARNING",
        "ERROR",
        "ERROR", "ERROR",
        "WARNING",
        "ERROR"
    ],
    "alarm_time": [
        "2026-08-20 08:00:00",
        "2026-08-20 09:00:00",
        "2026-08-20 10:00:00",
        "2026-08-20 11:00:00",
        "2026-08-20 12:00:00",
        "2026-08-20 13:00:00",
        "2026-08-20 14:00:00"
    ]
})

df_maintenance = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004],
    "device_id": ["R05", "R16", "R34", "R34"],
    "order_status": [
        "COMPLETED",
        "OPEN",
        "COMPLETED",
        "OPEN"
    ]
})

df_alarm["alarm_time"] = pd.to_datetime(df_alarm["alarm_time"])

df_device

,device_id,device_name
0,R05,前向散射仪
1,R16,云高仪
2,R34,跑道视程仪
3,R36,自动气象站
4,R40,温湿度传感器


## 题目要求

以设备表 `df_device` 为主表。

找出同时满足以下两个条件的设备：

```text
存在 ERROR 告警
并且
不存在 COMPLETED 维修工单
```

---

## 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `device_name` | 设备名称 |


In [2]:
df_alarm

,alarm_id,device_id,alarm_level,alarm_time
0,101,R05,ERROR,2026-08-20 08:00:00
1,102,R05,WARNING,2026-08-20 09:00:00
2,201,R16,ERROR,2026-08-20 10:00:00
3,301,R34,ERROR,2026-08-20 11:00:00
4,302,R34,ERROR,2026-08-20 12:00:00
5,401,R36,WARNING,2026-08-20 13:00:00
6,501,R40,ERROR,2026-08-20 14:00:00


In [3]:
df_maintenance

,order_id,device_id,order_status
0,1001,R05,COMPLETED
1,1002,R16,OPEN
2,1003,R34,COMPLETED
3,1004,R34,OPEN


In [8]:
query = '''
SELECT
    dd.device_id,
    dd.device_name
FROM df_device AS dd
WHERE EXISTS (
    SELECT 1
    FROM df_alarm AS da
    WHERE dd.device_id = da.device_id
      AND da.alarm_level = 'ERROR'
)
AND NOT EXISTS (
    SELECT 1
    FROM df_maintenance AS dm
    WHERE dd.device_id = dm.device_id
      AND dm.order_status = 'COMPLETED'
)
ORDER BY dd.device_id;
'''
df = duckdb.execute(query).fetchdf()
df

,device_id,device_name
0,R16,云高仪
1,R40,温湿度传感器
